In [1]:
import pandas as pd
df_network = pd.read_csv("problem1.csv")

Cost = {}
for i in df_network.index:
    Cost[(df_network['from'][i],df_network['to'][i])] = df_network['cost ($/bbl)'][i]

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
# define supply/demand for each node:
Supply = { 1: -6000, 2: -4500, 3: 0, 4: 3500, 5: 4000, 6: 3000 }

In [3]:
Cost

{(1, 3): 1.7,
 (1, 4): 4.5,
 (1, 5): 4.6,
 (1, 6): 4.1,
 (2, 3): 1.9,
 (2, 4): 4.3,
 (2, 5): 4.5,
 (2, 6): 4.1,
 (3, 4): 1.8,
 (3, 5): 1.6}

In [4]:
Nodes = Supply.keys()
Arcs = Cost.keys()

In [15]:
from docplex.mp.model import Model
mdl = Model()

# variables
shipped = mdl.continuous_var_dict(Arcs, lb=0, name='shipped')

In [16]:
# objective
mdl.minimize(mdl.sum(Cost[i,j]*shipped[i,j] for (i,j) in Arcs))

In [17]:
# flow conservation constraints
for j in Nodes:
    inflow = mdl.sum(shipped[i,j] for i in Nodes if (i,j) in Arcs)
    outflow = mdl.sum(shipped[j,i] for i in Nodes if (j,i) in Arcs)
    mdl.add_constraint(inflow - outflow == Supply[j])

In [18]:
# max throughput for site 3
#mdl.add_constraint(mdl.sum(shipped[i,3] for i in Nodes if (i,3) in Arcs) <= 3000)
mdl.add_constraint(mdl.sum(shipped[i,j] for (i,j) in Arcs if j==3) <=3000)

docplex.mp.LinearConstraint[](shipped_1_3+shipped_2_3,LE,3000)

In [19]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.00230503,status='optimal')

In [10]:
mdl.print_solution()

objective: 41750.000
status: OPTIMAL_SOLUTION(2)
  shipped_1_3=3000.000
  shipped_1_6=3000.000
  shipped_2_4=3500.000
  shipped_2_5=1000.000
  shipped_3_5=3000.000


In [20]:
mdl.print_solution()

objective: 41750.000
status: OPTIMAL_SOLUTION(2)
  shipped_1_3=3000.000
  shipped_1_6=3000.000
  shipped_2_4=3500.000
  shipped_2_5=1000.000
  shipped_3_5=3000.000
